In [67]:
import h5py

f = h5py.File('nueArCC_sns_yDir_g4_0000.h5', 'r')
print(f['segments'].dtype)
print(f['segments'][1437])
print(f['segments'][1472])

for traj in f['segments'][1437:1473]:
    print(traj['traj_id'])

{'names': ['event_id', 'vertex_id', 'file_vertex_id', 'segment_id', 'z_end', 'traj_id', 'file_traj_id', 'tran_diff', 'z_start', 'x_end', 'y_end', 'n_electrons', 'pdg_id', 'x_start', 'y_start', 't_start', 't0_start', 't0_end', 't0', 'dx', 'long_diff', 'pixel_plane', 't_end', 'dEdx', 'dE', 't', 'y', 'x', 'z', 'n_photons'], 'formats': ['<u4', '<u8', '<u8', '<u4', '<f4', '<i4', '<u4', '<f4', '<f4', '<f4', '<f4', '<u4', '<i4', '<f4', '<f4', '<f4', '<f8', '<f8', '<f8', '<f4', '<f4', '<i4', '<f4', '<f4', '<f4', '<f4', '<f4', '<f4', '<f4', '<f4'], 'offsets': [0, 8, 16, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 80, 88, 96, 104, 108, 112, 116, 120, 124, 128, 132, 136, 140, 144], 'itemsize': 152}
(19, 0, 19, 1437, -50.915752, 17, 849, 0.0, -50.88173, -34.810642, 52.042625, 0, 22, -34.792038, 52.074192, 0.0, 0.0037347404291393454, 0.0037632083238083335, 0.0037489743764738395, 0.050001547, 0.0, 0, 0.0, 0.808522, 0.04042735, 0.0, 52.05841, -34.80134, -50.898743, 0.0)
(19, 0, 19, 1472, -11.

In [70]:
import h5py

f = h5py.File('nueArCC_sns_yDir_g4_0000.h5', 'r')
print(f['trajectories'].dtype)
print(f['trajectories'][835])

{'names': ['event_id', 'vertex_id', 'file_vertex_id', 'traj_id', 'file_traj_id', 'parent_id', 'primary', 'E_start', 'pxyz_start', 'xyz_start', 't_start', 'E_end', 'pxyz_end', 'xyz_end', 't_end', 'pdg_id', 'start_process', 'start_subprocess', 'end_process', 'end_subprocess', 'dist_travel'], 'formats': ['<u4', '<u8', '<u8', '<i4', '<u4', '<i4', '?', '<f4', ('<f4', (3,)), ('<f4', (3,)), '<f8', '<f4', ('<f4', (3,)), ('<f4', (3,)), '<f8', '<i4', '<i4', '<i4', '<i4', '<i4', '<f4'], 'offsets': [0, 8, 16, 24, 28, 32, 36, 40, 44, 56, 72, 80, 84, 96, 112, 120, 124, 128, 132, 136, 140], 'itemsize': 144}
(19, 0, 19, 3, 835, -1, True, 1.6126766, [0.398, -0.789, -1.349], [24.353159, 18.813675, -11.592094], 0.0, 1.5093763e-08, [-0.0, 0.0, 0.0], [9.696472, -4.6469803, -68.55461], 0.003403051228526674, 22, 2, 13, 2, 12, 102.02091)


In [6]:
import h5py

# This function ensures that `event_id` is monotonically increasing with the vertex number
def check_vertices_well_formed(f):
    print("Checking that vertices are well formed...")
    event_id = -1
    
    for vertex in f['vertices']:
        if vertex['event_id'] < event_id:
            print("Bad vertex: vertex #" + str(vertex['file_vertex_id']) + " is misplaced!")
        event_id = vertex['event_id']

    print("Done!\n")

# This function ensures that each event has exactly one vertex (test requires that vertices are well formed)
def check_vertices(f):
    print("Checking vertices...")
    
    event_id = -1
    for vertex in f['vertices']:
        if vertex['vertex_id'] != 0:
            print("Bad vertex: vertex #" + str(vertex['file_vertex_id']) + " has vertex ID " + str(vertex['vertex_id']) + " (event ID: " + str(vertex['event_id']) + ")!")
        
        while vertex['event_id'] > (event_id + 1):
            print("Bad event: event #" + str(event_id + 1) + " has no associated vertex!")
            event_id += 1

        event_id = vertex['event_id']
    
    print("Done!\n")

f = h5py.File('nueArCC_sns_yDir_g4_0000.h5', 'r')
check_vertices(f)

Checking vertices...
Done!



In [68]:
import h5py

# This function ensures that `event_id` is monotonically increasing with trajectory number
def check_trajectories_well_formed(f):
    print("Checking that trajectories are well formed...")
    event_id = -1

    for traj in f['trajectories']:
        if traj['event_id'] < event_id:
            print("Bad trajectory: trajectory #" + str(traj['file_traj_id']) + " is misplaced!")
        event_id = traj['event_id']

    print("Done!\n")

# This function ensures that all events have no duplicate trajectories and all trajectories are present (i.e., trajectories with IDs 0 to (n-1), where n is the number of trajectories for that event)
def check_trajectories(f):
    print("Checking trajectories...")
    event_id = -1
    traj_ids = []
    
    for traj in f['trajectories']:
        if traj['event_id'] > event_id:
            for i in range(len(traj_ids)):
                if i not in traj_ids:
                    print("Bad event: event #" + str(event_id) + " has no trajectory #" + str(i) + "!")
                    
            event_id = traj['event_id']
            traj_ids = []

        if traj['traj_id'] in traj_ids:
            print("Bad trajectory: trajectory #" + str(traj['file_traj_id']) + " with trajectory ID " + str(traj['traj_id']) + " is a duplicate (event ID: " + str(traj['event_id']) + ")!")
        
        traj_ids.append(traj['traj_id'])

    print("Done!\n")

def check_for_duplicate_hit_trajectories(f):
    print("Checking for duplicate hit trajectories...")

    event_id = 0
    hit = False
    
    for traj in f['trajectories']:
        if traj['event_id'] > event_id:
            if not hit:
                print("Bad event: event #" + str(event_id) + " has no hit trajectories!")
            
            hit = False
            event_id = traj['event_id']
            
        if traj['parent_id'] == -1 and traj['pdg_id'] == 11:
            if hit:
                print("Bad event: event #" + str(event_id) + " has at least hit trajectories!")
            else:
                hit = True

    print("Done!\n")

f = h5py.File('nueArCC_sns_yDir_g4_0000.h5', 'r')
check_trajectories(f)

Checking trajectories...
Bad event: event #19 has no trajectory #25!
Bad event: event #39 has no trajectory #49!
Bad event: event #105 has no trajectory #15!
Bad event: event #107 has no trajectory #21!
Bad event: event #119 has no trajectory #55!
Bad event: event #195 has no trajectory #10!
Bad event: event #315 has no trajectory #19!
Bad event: event #336 has no trajectory #22!
Bad event: event #457 has no trajectory #58!
Bad event: event #630 has no trajectory #11!
Bad event: event #639 has no trajectory #10!
Bad event: event #788 has no trajectory #41!
Bad event: event #851 has no trajectory #38!
Bad event: event #943 has no trajectory #7!
Bad event: event #1086 has no trajectory #24!
Bad event: event #1121 has no trajectory #11!
Bad event: event #1168 has no trajectory #29!
Bad event: event #1249 has no trajectory #9!
Bad event: event #1284 has no trajectory #6!
Bad event: event #1339 has no trajectory #26!
Bad event: event #1570 has no trajectory #23!
Bad event: event #1594 has n

In [ ]:
import h5py

# This function ensures that `event_id` is monotonically increasing with segment number
def check_segments_well_formed(f):
    print("Checking that segments are well formed...")
    event_id = -1

    for seg in f['segments']:
        if seg['event_id'] < event_id:
            print("Bad segment: segment #" + str(seg['segment_id']) + " is misplaced!")
        event_id = seg['event_id']

    print("Done!\n")

# This function ensures that the energy deposition of each segment sums to the total energy of the trajectory
def check_segments(f):
    print("Checking segments...")
    
    event_id = -1
    energies = {}
    for seg in f['segments']:
        if seg['event_id'] > event_id:
            for file_traj_id in energies:
                if energies[file_traj_id] != (f['trajectories'][file_traj_id]['E_start'] - f['trajectories'][file_traj_id]['E_end']):
                    print("Bad trajectory: segment energies corresponding to trajectory #" + str(file_traj_id) + " (traj_id: " + str(f['trajectories'][file_traj_id]['traj_id']) + ", event_id: " + str(f['trajectories'][file_traj_id]['event_id']) + ") do not match! The sum of the segment energies is " + str(energies[file_traj_id]) + ", while the difference in trajectory energies is " + str(f['trajectories'][file_traj_id]['E_start'] - f['trajectories'][file_traj_id]['E_end']) + "!")
            energies = {}
            
        event_id = seg['event_id']

        if energies.get(seg['file_traj_id']):
            energies[seg['file_traj_id']] += seg['dE']
        else:
            energies[seg['file_traj_id']] = seg['dE']

    print("Done!\n")
            


f = h5py.File('/sdf/data/neutrino/yuntse/coherent/SNeNDSens/g4/NueArCC/nueArCC_sns_yDir_g4_0000.h5', 'r')